In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_genre_df = (
    spark.read
        .format("json")
        .option("header", "true")
        .load(f"{bronze_folder_path}/movie_genre.json")
)
movie_genre_df.show()

+-------+-------+
|genreId|movieId|
+-------+-------+
|     35|      5|
|     80|      5|
|     12|     11|
|     28|     11|
|    878|     11|
|     16|     12|
|  10751|     12|
|     18|     13|
|     35|     13|
|  10749|     13|
|     18|     14|
|     18|     16|
|     80|     16|
|  10402|     16|
|     12|     18|
|     14|     18|
|     28|     18|
|     53|     18|
|    878|     18|
|     18|     19|
+-------+-------+
only showing top 20 rows


In [0]:
movie_genre_shema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("movieId", IntegerType(), True)
])
movie_genre_df = (
    spark.read
        .format("json")
        .schema(movie_genre_shema)
        .load(f"{bronze_folder_path}/movie_genre.json")
)
movie_genre_df.printSchema()
display(movie_genre_df)

root
 |-- genreId: integer (nullable = true)
 |-- movieId: integer (nullable = true)



genreId,movieId
35,5
80,5
12,11
28,11
878,11
16,12
10751,12
18,13
35,13
10749,13


In [0]:
movie_genre_df_final = (
    movie_genre_df
    .withColumnsRenamed({"genreId": "genre_id", "movieId": "movie_id"})
    .withColumns({"env": lit(v_env)})
)

movie_genre_df_final = add_ingestion_date(movie_genre_df_final)

display(movie_genre_df)

genreId,movieId
35,5
80,5
12,11
28,11
878,11
16,12
10751,12
18,13
35,13
10749,13


In [0]:
(
    movie_genre_df_final
    .write
    .format("parquet")
    .mode("overwrite")
    #.partitionBy("movie_id")
    .save(f"{silver_folder_path}/movie_genre")
)

In [0]:
dbutils.notebook.exit("Success")